# Week 4 · Day 1 — RAG Basics (Testing Notebook)

**RAG = Retrieval Augmented Generation** — make the AI answer using *your* documents.

This notebook walks through the 4 core concepts hands-on:

1. **Embeddings** — text → vectors (SentenceTransformers)
2. **Vector Database** — store embeddings (ChromaDB)
3. **Semantic Search** — find relevant chunks
4. **LLM Response Generation** — generate a grounded answer (optional)

It ships with built-in sample notes, so it runs **offline end-to-end** (only the final LLM step needs an `OPENAI_API_KEY`).

> Install first: `pip install -r ../backend/requirements-rag.txt`

## 0. Check the libraries are installed

In [1]:
import importlib

for pkg in ["sentence_transformers", "chromadb", "langchain", "pypdf"]:
    try:
        m = importlib.import_module(pkg)
        print(f"OK   {pkg:<22} {getattr(m, '__version__', '?')}")
    except Exception as e:  # pragma: no cover
        print(f"MISSING {pkg:<19} -> pip install -r ../backend/requirements-rag.txt ({e})")

/Users/tharmithan.s/Desktop/MentorMindAi/backend/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OK   sentence_transformers  5.5.1


OK   chromadb               1.5.9
OK   langchain              1.3.2


OK   pypdf                  6.12.2


## 1. Embeddings — text → vectors

An embedding turns text into a vector of numbers that captures its *meaning*. Similar meaning → similar (close) vectors. We use `all-MiniLM-L6-v2` (384 dims, small + fast + local).

In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Downloads once (~80MB), then cached locally.
embedder = SentenceTransformer("all-MiniLM-L6-v2")

vec = embedder.encode("I love studying algebra")
print("Vector dimensions:", vec.shape)
print("First 8 numbers:", np.round(vec[:8], 3))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17308.23it/s]

Vector dimensions: (384,)
First 8 numbers: [-0.009  0.028 -0.051  0.029 -0.063 -0.07   0.014  0.015]


In [3]:
# Semantic similarity: close meaning -> high cosine similarity (max 1.0)
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

sentences = [
    "I love studying algebra",
    "Maths is really fun",
    "The cat slept on the warm sofa",
]
emb = embedder.encode(sentences)

print(f"'{sentences[0]}'  vs  '{sentences[1]}'  -> {cosine(emb[0], emb[1]):.3f}  (similar)")
print(f"'{sentences[0]}'  vs  '{sentences[2]}'  -> {cosine(emb[0], emb[2]):.3f}  (different)")

'I love studying algebra'  vs  'Maths is really fun'  -> 0.689  (similar)
'I love studying algebra'  vs  'The cat slept on the warm sofa'  -> 0.028  (different)


## 2 + 3. Vector Database & Semantic Search

Now the real RAG flow on a small set of study notes:

1. **Split** the document into chunks (LangChain).
2. **Embed** each chunk and **store** it in ChromaDB.
3. **Search** by meaning to retrieve the most relevant chunks for a question.

In [4]:
# Sample study notes (stand-in for an uploaded PDF).
study_notes = """
Photosynthesis is the process plants use to convert sunlight, water, and carbon dioxide
into glucose and oxygen. It happens in the chloroplasts, mainly in the leaves.

Newton's second law of motion states that force equals mass times acceleration (F = ma).
It explains how the velocity of an object changes when a force is applied.

The water cycle describes how water evaporates from oceans, condenses into clouds,
and falls back as rain or snow. The main stages are evaporation, condensation,
precipitation, and collection.

Effective exam preparation involves spaced repetition, active recall, and regular sleep.
Cramming the night before is far less effective than studying a little each day.
"""

In [5]:
# 1) Split into chunks with overlap (so context isn't cut mid-idea).
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=40)
chunks = splitter.split_text(study_notes)

print(f"Split into {len(chunks)} chunks:\n")
for i, c in enumerate(chunks):
    print(f"[{i}] {c.strip()[:90]}...")

Split into 4 chunks:

[0] Photosynthesis is the process plants use to convert sunlight, water, and carbon dioxide
in...
[1] Newton's second law of motion states that force equals mass times acceleration (F = ma).
I...
[2] The water cycle describes how water evaporates from oceans, condenses into clouds,
and fal...
[3] Effective exam preparation involves spaced repetition, active recall, and regular sleep.
C...


In [6]:
# 2) Embed each chunk and store in ChromaDB.
import chromadb

client = chromadb.EphemeralClient()  # in-memory; use PersistentClient(path=...) to keep on disk
collection = client.get_or_create_collection("study_notes")

chunk_embeddings = embedder.encode(chunks).tolist()
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=chunk_embeddings,
)
print(f"Stored {collection.count()} chunks in ChromaDB.")

Stored 4 chunks in ChromaDB.


In [7]:
# 3) Semantic search: embed the question, find nearest chunks.
def retrieve(question, k=2):
    q_emb = embedder.encode(question).tolist()
    res = collection.query(query_embeddings=[q_emb], n_results=k)
    return res["documents"][0], res["distances"][0]

question = "How do plants make food?"
docs, dists = retrieve(question)

print(f"Q: {question}\n")
for d, dist in zip(docs, dists):
    print(f"(distance {dist:.3f})  {d.strip()}\n")

Q: How do plants make food?

(distance 0.957)  Photosynthesis is the process plants use to convert sunlight, water, and carbon dioxide
into glucose and oxygen. It happens in the chloroplasts, mainly in the leaves.

(distance 1.686)  The water cycle describes how water evaporates from oceans, condenses into clouds,
and falls back as rain or snow. The main stages are evaporation, condensation,
precipitation, and collection.



In [8]:
# Notice it matches by MEANING, not keywords: 'make food' -> photosynthesis chunk.
for q in ["What is the best way to revise for tests?", "Explain force and acceleration"]:
    docs, _ = retrieve(q, k=1)
    print(f"Q: {q}\n -> {docs[0].strip()}\n")

Q: What is the best way to revise for tests?
 -> Effective exam preparation involves spaced repetition, active recall, and regular sleep.
Cramming the night before is far less effective than studying a little each day.

Q: Explain force and acceleration
 -> Newton's second law of motion states that force equals mass times acceleration (F = ma).
It explains how the velocity of an object changes when a force is applied.



## 4. LLM Response Generation (optional)

Finally, feed the retrieved chunks + question into an LLM to write a grounded answer. This step needs an OpenAI-compatible API key. Without one, we just print the prompt that *would* be sent.

In [9]:
def build_prompt(question, k=2):
    docs, _ = retrieve(question, k=k)
    context = "\n".join(f"- {d.strip()}" for d in docs)
    return (
        "Use ONLY the context below to answer the question. "
        "If the answer isn't in the context, say you don't know.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )

prompt = build_prompt("How do plants make food?")
print(prompt)

Use ONLY the context below to answer the question. If the answer isn't in the context, say you don't know.

Context:
- Photosynthesis is the process plants use to convert sunlight, water, and carbon dioxide
into glucose and oxygen. It happens in the chloroplasts, mainly in the leaves.
- The water cycle describes how water evaporates from oceans, condenses into clouds,
and falls back as rain or snow. The main stages are evaporation, condensation,
precipitation, and collection.

Question: How do plants make food?
Answer:


In [10]:
import os

def answer(question):
    api_key = os.getenv("OPENAI_API_KEY")
    prompt = build_prompt(question)
    if not api_key:
        return "[No OPENAI_API_KEY set — skipping LLM call. Prompt is ready above.]"
    try:
        from openai import OpenAI
        client = OpenAI(api_key=api_key)
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        return resp.choices[0].message.content
    except Exception as e:  # pragma: no cover
        return f"[LLM call failed: {e}]"

print(answer("How do plants make food?"))

[No OPENAI_API_KEY set — skipping LLM call. Prompt is ready above.]


## Recap

| Step | Tool | Result |
|------|------|--------|
| Embed | SentenceTransformers | text → 384-dim vectors |
| Split | LangChain splitter | document → overlapping chunks |
| Store / Search | ChromaDB | meaning-based retrieval |
| Generate | LLM (optional) | grounded natural-language answer |

**Next (Day 2):** wire this into the backend — `/rag/upload` to index PDFs and `/rag/ask` to answer questions about them.